# Methods Pipeline — Swim-Lane Diagram

Renders the 20-phase ComputationalReview pipeline as a swim-lane diagram with 4 distinct roles, faithfully reproducing the actor/critic separation defined in `skills/comprev-orchestrator-v26.md`.

**Lanes (top to bottom):**
- **Coordinator** — Phase 1 only; gates between phases (dotted rail)
- **DATAML agent** — mechanical / programmatic phases (3, 5, 9, 13, 14, 15, 17, 19, 20)
- **LITREVIEW agent** — scientific judgment phases (2, 4, 6, 7, 8, 10, 11, 12, 16, 18)

LITREVIEW phases acting as **blinded critics** (information-barriered from the actor) are hatched and bordered in purple: phases 6 (Figure Audit), 8 (Section Critics), 12 (Bookend Critic).

This figure replaces an earlier row diagram that incorrectly showed the Coordinator managing DATAML phases and merged "LITREVIEW/DATAML" in the legend as if interchangeable.


In [ ]:
"""
Figure: Methods Pipeline (swim-lane diagram)
============================================

Renders the 20-phase ComputationalReview pipeline as a swim-lane diagram with
4 distinct roles, faithfully reproducing the actor/critic separation defined
in skills/comprev-orchestrator-v26.md.

Lanes (top to bottom):
  - Coordinator    -- Phase 1 only; gates between phases (dotted rail)
  - DATAML agent   -- mechanical/programmatic phases (3, 5, 9, 13, 14, 15, 17, 19, 20)
  - LITREVIEW agent -- scientific judgment phases (2, 4, 6, 7, 8, 10, 11, 12, 16, 18)

LITREVIEW phases that act as BLINDED CRITICS (information-barriered from the
actor) are hatched and bordered in purple: phases 6 (Figure Audit),
8 (Section Critics), 12 (Bookend Critic).

This figure replaces an earlier row diagram that incorrectly showed the
Coordinator owning DATAML phases and merged "LITREVIEW/DATAML" in the legend
as if interchangeable.

Output: figures/fig_methods_pipeline.png
"""
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# Authoritative phase table (per comprev-orchestrator-v26.md)
# (phase_number, label, role, is_blinded_critic)
PHASES = [
    ( 1, "Scope",                 "coord",  False),
    ( 2, "Evidence\nGathering",   "expert", False),
    ( 3, "Citation\nInfra",       "dataml", False),
    ( 4, "Scaffold",              "expert", False),
    ( 5, "Evidence\nCuration",    "dataml", False),
    ( 6, "Figure\nAudit",         "expert", True ),
    ( 7, "Section\nDrafting",     "expert", False),
    ( 8, "Section\nCritics",      "expert", True ),
    ( 9, "Bibliography",          "dataml", False),
    (10, "Integration",           "expert", False),
    (11, "Intro /\nConclusion",   "expert", False),
    (12, "Bookend\nCritic",       "expert", True ),
    (13, "Methods",               "dataml", False),
    (14, "Document\nAssembly",    "dataml", False),
    (15, "Citation\nTriples",     "dataml", False),
    (16, "Citation\nVerification","expert", False),
    (17, "Fix\nPreparation",      "dataml", False),
    (18, "Fix\nExecution",        "expert", False),
    (19, "Fix\nApplication",      "dataml", False),
    (20, "Repository\nPush",      "dataml", False),
]

EDGE = {"coord": "#2e7d32", "dataml": "#1565c0", "expert": "#e65100"}
FILL = {"coord": "#c8e6c9", "dataml": "#bbdefb", "expert": "#ffe0b2"}
CRITIC_EDGE = "#6a1b9a"
CRITIC_HATCH = "////"

LANE_Y = {"coord": 3.2, "dataml": 1.7, "expert": 0.2}
LANE_LABEL = {
    "coord":  "COORDINATOR\n(Phase 1 + gates)",
    "dataml": "DATAML AGENT\n(mechanical)",
    "expert": "LITREVIEW AGENT\n(scientific)",
}

fig, ax = plt.subplots(figsize=(17, 5.5))
ax.set_xlim(-2.0, 21.4)
ax.set_ylim(-0.7, 4.6)
ax.axis("off")

# Lane backgrounds
for role, y in LANE_Y.items():
    ax.add_patch(FancyBboxPatch(
        (-1.5, y - 0.55), 22.5, 1.10,
        boxstyle="round,pad=0.02",
        facecolor=FILL[role], edgecolor="none", alpha=0.20, zorder=0,
    ))
    ax.text(-1.4, y, LANE_LABEL[role],
            ha="left", va="center", fontsize=9, weight="bold",
            color=EDGE[role], zorder=1)

# Coordinator gate rail
GATE_Y = LANE_Y["coord"] - 0.62
ax.plot([0.5, 20.5], [GATE_Y, GATE_Y], color="#8a8a8a",
        linewidth=0.7, linestyle=(0, (1, 2)), zorder=1)
ax.text(20.7, GATE_Y, "  gate", ha="left", va="center", fontsize=7,
        color="#666", style="italic")

# Inter-phase arrows (drawn first so boxes overlay them)
box_w, box_h = 0.95, 0.88
for i in range(len(PHASES) - 1):
    x1, y1 = PHASES[i][0], LANE_Y[PHASES[i][2]]
    x2, y2 = PHASES[i+1][0], LANE_Y[PHASES[i+1][2]]
    same_lane = (PHASES[i][2] == PHASES[i+1][2])
    rad = 0.0 if same_lane else (-0.18 if y2 < y1 else 0.18)
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=0.9, color="#888",
                                connectionstyle=f"arc3,rad={rad}",
                                shrinkA=18, shrinkB=18),
                zorder=2)

# Phase boxes
for n, label, role, blinded in PHASES:
    x = n
    y = LANE_Y[role]
    fc = FILL[role]
    ec = CRITIC_EDGE if blinded else EDGE[role]
    lw = 2.0 if blinded else 1.5
    ax.add_patch(FancyBboxPatch(
        (x - box_w/2, y - box_h/2), box_w, box_h,
        boxstyle="round,pad=0.03",
        facecolor=fc, edgecolor=ec, linewidth=lw,
        hatch=CRITIC_HATCH if blinded else None,
        zorder=3,
    ))
    ax.text(x, y, label, ha="center", va="center", fontsize=8.4,
            zorder=4, weight="bold" if blinded else "regular")
    ax.text(x, y + box_h/2 + 0.08, str(n),
            ha="center", va="bottom", fontsize=8.5, weight="bold",
            color=EDGE[role], zorder=4)
    ax.plot([x, x], [LANE_Y["coord"] + box_h/2 - 0.5, GATE_Y],
            color="#bbb", linewidth=0.5, linestyle=":", zorder=1)

ax.text(20, LANE_Y["dataml"] - 0.65,
        "(Phase 20a Methods Ledger Refresh runs before 20)",
        ha="right", va="top", fontsize=6.8, color="#666", style="italic")

# Legend
legend_items = [
    mpatches.Patch(facecolor=FILL["coord"], edgecolor=EDGE["coord"],
                   label="Coordinator -- owns Phase 1 + gates between phases"),
    mpatches.Patch(facecolor=FILL["dataml"], edgecolor=EDGE["dataml"],
                   label="DATAML agent -- mechanical / programmatic phases"),
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=EDGE["expert"],
                   label="LITREVIEW agent -- scientific judgment phases"),
    mpatches.Patch(facecolor=FILL["expert"], edgecolor=CRITIC_EDGE,
                   hatch=CRITIC_HATCH,
                   label="LITREVIEW blinded critic -- information barrier from actor"),
]
ax.legend(handles=legend_items, loc="lower center",
          bbox_to_anchor=(0.5, -0.10), ncol=2,
          frameon=False, fontsize=9, handlelength=2.0, handleheight=1.1)

ax.set_title("Expert Review Pipeline v26 -- actor-critic separation across 20 phases",
             fontsize=12, weight="bold", pad=10)

plt.subplots_adjust(left=0.02, right=0.99, top=0.92, bottom=0.12)

# When run from figures/notebooks/, write PNG one level up
out_path = os.path.join(os.path.dirname(os.path.abspath("__nb__")), "..", "fig_methods_pipeline.png")
fig.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
print("saved:", out_path)
